# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/palkinsuneja/palkin-flyrank-ml-internship-july-to-sept-2026/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content for refresh when it shows a combination of search opportunity and weak engagement.

The rule gives a higher action score to content with higher search impressions, weaker CTR relative to its search visibility, and lower average position. The score is used only to prioritize pages for review, not to claim that a refresh will cause improvement.

### Reason codes

- `HIGH_IMPRESSIONS` — the content receives meaningful search visibility.
- `LOW_CTR` — clicks are low relative to impressions.
- `WEAK_POSITION` — average search position indicates room for improvement.
- `MULTI_SIGNAL` — more than one of the above signals is present.

In [6]:
# Section 1: My rule and its reason codes

import os
import duckdb
import pandas as pd
from google.colab import userdata

# Get Hugging Face token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Use February 2026 as the decision-time feature window
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

# Aggregate February signals to content-item level
rule_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)
            AS gsc_avg_position
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    LIMIT 10000
""").df()

# Score: higher priority for visible content with weak performance
rule_df["score"] = 0

rule_df.loc[
    (rule_df["gsc_impressions"] >= 1000) &
    (rule_df["gsc_avg_position"] > 10) &
    (rule_df["ctr"] < 0.02),
    "score"
] = 3

rule_df.loc[
    (rule_df["gsc_impressions"] >= 1000) &
    (rule_df["gsc_avg_position"] > 10) &
    (rule_df["ctr"] >= 0.02),
    "score"
] = 2

rule_df.loc[
    (rule_df["gsc_impressions"] < 1000) &
    (rule_df["gsc_avg_position"] > 10),
    "score"
] = 1

# Reason codes
rule_df.loc[
    (rule_df["gsc_impressions"] >= 1000) &
    (rule_df["gsc_avg_position"] > 10) &
    (rule_df["ctr"] < 0.02),
    "reason_code"
] = "HIGH_VISIBILITY_WEAK_PERFORMANCE"

rule_df.loc[
    (rule_df["gsc_impressions"] >= 1000) &
    (rule_df["gsc_avg_position"] > 10) &
    (rule_df["ctr"] >= 0.02),
    "reason_code"
] = "HIGH_VISIBILITY_POOR_POSITION"

rule_df.loc[
    (rule_df["gsc_impressions"] < 1000) &
    (rule_df["gsc_avg_position"] > 10),
    "reason_code"
] = "POOR_POSITION_LOW_VOLUME"

# Action labels
rule_df["action"] = rule_df["score"].map({
    3: "REFRESH_NOW",
    2: "REVIEW",
    1: "REVIEW",
    0: "NO_ACTION"
})

# Rank highest-priority content first
rule_df = rule_df.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

print("Rule created successfully.")
print("Rows:", len(rule_df))
display(rule_df.head(10))

Rule created successfully.
Rows: 10000


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
0,client_fef1a8f436438636,content_84a6bf3578312e90,79986.0,74.0,0.000925,19.783062,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
1,client_fef1a8f436438636,content_ba462518dad435fc,69134.0,39.0,0.000564,27.477479,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
2,client_fef1a8f436438636,content_7a0a59b4cb181ab9,40362.0,34.0,0.000842,35.594569,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
3,client_fef1a8f436438636,content_ae68e15ceab3a802,38744.0,204.0,0.005265,10.713736,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
4,client_fef1a8f436438636,content_fd358a60ca37c05f,28362.0,37.0,0.001305,17.022389,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
5,client_fef1a8f436438636,content_510b0e6411887145,24577.0,155.0,0.006307,12.971640,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
6,client_fef1a8f436438636,content_6b5c4c849789343f,21655.0,24.0,0.001108,13.685292,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
7,client_fef1a8f436438636,content_f5bf7728c2d5a382,17249.0,31.0,0.001797,10.995884,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
8,client_fef1a8f436438636,content_bedfef521fdf0527,16461.0,19.0,0.001154,15.784156,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW
9,client_fef1a8f436438636,content_cbab33f725fe2de2,15896.0,25.0,0.001573,15.096565,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the ranked action queue and save it as CSV

output_cols = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action"
]

baseline_action_score = rule_df[output_cols].copy()

# Rank by highest score first, then impressions
baseline_action_score = baseline_action_score.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_action_score["rank"] = (
    baseline_action_score.index + 1
)

# Save ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_action_score.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked queue created successfully.")
print("Rows:", len(baseline_action_score))
print("Saved to: work/outputs/baseline_action_score.csv")

display(baseline_action_score.head(20))

Ranked queue created successfully.
Rows: 10000
Saved to: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action,rank
0,client_fef1a8f436438636,content_84a6bf3578312e90,79986.0,74.0,0.000925,19.783062,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,1
1,client_fef1a8f436438636,content_ba462518dad435fc,69134.0,39.0,0.000564,27.477479,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,2
2,client_fef1a8f436438636,content_7a0a59b4cb181ab9,40362.0,34.0,0.000842,35.594569,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,3
3,client_fef1a8f436438636,content_ae68e15ceab3a802,38744.0,204.0,0.005265,10.713736,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,4
4,client_fef1a8f436438636,content_fd358a60ca37c05f,28362.0,37.0,0.001305,17.022389,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,5
5,client_fef1a8f436438636,content_510b0e6411887145,24577.0,155.0,0.006307,12.971640,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,6
6,client_fef1a8f436438636,content_6b5c4c849789343f,21655.0,24.0,0.001108,13.685292,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,7
7,client_fef1a8f436438636,content_f5bf7728c2d5a382,17249.0,31.0,0.001797,10.995884,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,8
8,client_fef1a8f436438636,content_bedfef521fdf0527,16461.0,19.0,0.001154,15.784156,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,9
9,client_fef1a8f436438636,content_cbab33f725fe2de2,15896.0,25.0,0.001573,15.096565,3,HIGH_VISIBILITY_WEAK_PERFORMANCE,REFRESH_NOW,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 rows are the highest-priority items produced by the baseline rule.  
For each item, I record the action, reason code, a simple confidence note, and what could make the recommendation wrong.  
The review is based only on February 2026 signals available at decision time.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Review the top-20 ranked actions

top20_review = baseline_action_score.head(20).copy()

def confidence_note(row):
    if row["gsc_impressions"] >= 1000 and row["gsc_avg_position"] > 10:
        return "HIGH"
    elif row["gsc_impressions"] >= 1000:
        return "MEDIUM"
    else:
        return "LOW"

def what_would_make_wrong(row):
    if row["gsc_impressions"] < 1000:
        return "Low search volume may make the signal unstable."
    elif row["gsc_avg_position"] <= 10:
        return "Good ranking would weaken the poor-position interpretation."
    else:
        return "Recent changes or data-quality issues could make the signal misleading."

top20_review["confidence"] = top20_review.apply(confidence_note, axis=1)
top20_review["what_would_make_wrong"] = top20_review.apply(
    what_would_make_wrong, axis=1
)

top20_review = top20_review[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence",
        "what_would_make_wrong"
    ]
]

print("Top-20 review created.")
display(top20_review)

Top-20 review created.


,rank,client_hash_id,content_hash_id,action,reason_code,confidence,what_would_make_wrong
0,1,client_fef1a8f436438636,content_84a6bf3578312e90,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
1,2,client_fef1a8f436438636,content_ba462518dad435fc,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
2,3,client_fef1a8f436438636,content_7a0a59b4cb181ab9,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
3,4,client_fef1a8f436438636,content_ae68e15ceab3a802,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
4,5,client_fef1a8f436438636,content_fd358a60ca37c05f,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
5,6,client_fef1a8f436438636,content_510b0e6411887145,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
6,7,client_fef1a8f436438636,content_6b5c4c849789343f,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
7,8,client_fef1a8f436438636,content_f5bf7728c2d5a382,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
8,9,client_fef1a8f436438636,content_bedfef521fdf0527,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...
9,10,client_fef1a8f436438636,content_cbab33f725fe2de2,REFRESH_NOW,HIGH_VISIBILITY_WEAK_PERFORMANCE,HIGH,Recent changes or data-quality issues could ma...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

Some high-ranked items may be weak picks when the underlying signal is based on low volume or borderline conditions.  
I also check that no future-window or label-derived fields are being used by the baseline rule.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Identify potentially weak picks and check for leakage

weak_picks = baseline_action_score[
    (baseline_action_score["gsc_impressions"] < 1000) |
    (baseline_action_score["gsc_avg_position"] <= 10)
].head(10).copy()

print("Potentially weak picks:")
display(
    weak_picks[
        [
            "rank",
            "gsc_impressions",
            "gsc_avg_position",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

# Leakage check: future/label-derived fields must not be used as inputs
forbidden_fields = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

used_fields = set(baseline_action_score.columns)
leaked_fields = sorted(forbidden_fields.intersection(used_fields))

print("\nLeakage check:")
if leaked_fields:
    print("WARNING - label/future-derived fields present:", leaked_fields)
else:
    print("PASS - no label-derived or future-window fields are used in the baseline queue.")

print("\nFeature columns used by the rule:")
display(
    baseline_action_score[
        [
            "gsc_impressions",
            "gsc_avg_position",
            "ctr"
        ]
    ].head()
)

Potentially weak picks:


,rank,gsc_impressions,gsc_avg_position,ctr,score,reason_code,action
639,640,996.0,12.502008,0.003012,1,POOR_POSITION_LOW_VOLUME,REVIEW
640,641,995.0,14.104523,0.003015,1,POOR_POSITION_LOW_VOLUME,REVIEW
641,642,994.0,19.006036,0.000000,1,POOR_POSITION_LOW_VOLUME,REVIEW
642,643,993.0,17.793555,0.003021,1,POOR_POSITION_LOW_VOLUME,REVIEW
643,644,993.0,19.162135,0.000000,1,POOR_POSITION_LOW_VOLUME,REVIEW
644,645,992.0,15.717742,0.000000,1,POOR_POSITION_LOW_VOLUME,REVIEW
645,646,991.0,10.722503,0.001009,1,POOR_POSITION_LOW_VOLUME,REVIEW
646,647,990.0,18.491919,0.000000,1,POOR_POSITION_LOW_VOLUME,REVIEW
647,648,990.0,10.050505,0.002020,1,POOR_POSITION_LOW_VOLUME,REVIEW
648,649,988.0,19.588057,0.002024,1,POOR_POSITION_LOW_VOLUME,REVIEW



Leakage check:
PASS - no label-derived or future-window fields are used in the baseline queue.

Feature columns used by the rule:


,gsc_impressions,gsc_avg_position,ctr
0,79986.0,19.783062,0.000925
1,69134.0,27.477479,0.000564
2,40362.0,35.594569,0.000842
3,38744.0,10.713736,0.005265
4,28362.0,17.022389,0.001305


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.